In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.impute import KNNImputer

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils import dados, pivot, salvar

In [ ]:
def filtro_colunas(df, min_missing, max_missing):
    """
    Seleciona colunas com dados faltantes entre min_missing e max_missing.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de interesse.
    min_missing : float
        Determina a fração mínima de dados faltantes tolerados numa coluna.
    max_missing : float
        Determina a fração máxima de dados faltantes tolerados numa coluna.

    Returns
    -------
    colunas_filtradas : list
        Lista com os nomes das colunas com dados faltantes entre min_missing e max_missing.
    """    
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    # Calcula a porcentagem de dados faltantes para cada coluna
    fracao_faltantes = df[variaveis].isna().mean()

    # Filtra colunas que estão no intervalo [min_missing, max_missing]
    return fracao_faltantes[(fracao_faltantes >= min_missing) & (fracao_faltantes <= max_missing)].index.tolist()

In [ ]:
# Teste: filtra colunas com dados faltantes entre 5% e 15% (para KNN)
df, metadados, variaveis = dados(r"data\input\dados_brutos.xlsx")
filtro_colunas(df, min_missing=0.05, max_missing=0.15)

['ALC.', 'AC.', 'Cianobacteria', 'Clorofila']

# Média

Imputação para colunas com <5% de dados faltantes

In [ ]:
def mean_imput(df, max_missing=0.05, 
               return_reduced=False, save_missing_pct=False, salvar_arquivo=False):
    """
    Faz a imputação dos dados faltantes via média.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de interesse.
    max_missing : float, default 0.05
        Determina a fração máxima de dados faltantes tolerados numa coluna.
        Se a fração de dados faltantes for maior que max_missing, o algoritmo de 
        imputação não será aplicado à coluna.
    return_reduced : bool, default False
        Se True, retorna apenas as colunas que receberam imputação.
    save_missing_pct : bool, default False
        Se True, retorna um dicionário com a porcentagem de dados faltantes 
        de cada coluna imputada.
    salvar_arquivo : bool, default False
        Determina se um arquivo Excel com o DataFrame pós imputação deve ser gerado.

    Returns
    -------
    df : pd.DataFrame
        DataFrame pós imputação.
    missing_pct : dict, optional
        Dicionário com as porcentagens de dados faltantes (apenas se save_missing_pct=True).
    """
    df = df.copy()

    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    cols = filtro_colunas(df, min_missing=0, max_missing=max_missing)
    
    # Calcula porcentagem de dados faltantes antes de imputar
    if save_missing_pct:
        missing_pct = df[cols].isna().mean().to_dict()
    
    # Faz a imputação
    df[cols] = df[cols].fillna(df[cols].mean())
    
    # Retorna um DataFrame sem as variáveis que não passaram pela imputação
    if return_reduced:
        # Lista os nomes das colunas desejadas no DataFrame final.
        colunas = set(cols + metadados)
        # Busca as colunas presentes na lista anterior.
        colunas = [col for col in df.columns if col in colunas]
        # Reduz o DataFrame às colunas de metadados + variáveis imputadas.
        df = df[colunas]
    
    if salvar_arquivo:
        salvar(df, "imput_mean")
    
    if save_missing_pct:
        return df, missing_pct
    
    return df

In [ ]:
# Teste
dados_limpos, metadados, variaveis = dados(r"data\input\dados_limpos.xlsx")

mean_imput(dados_limpos, return_reduced=True)

,Index,Data,data_normalizada,Estatistica,COR,TURB.,pH,O.C.,O.D.,Cl,DUR.,Fe,Mn,Cond.
0,0,2009-01-01,2009-01-01,Min.,200.000000,64.000000,7.000000,6.300000,2.000000,16.000000,36.000000,3.230000,0.090000,131.000000
1,1,2009-01-01,2009-01-01,Med.,432.000000,131.000000,7.200000,6.800000,3.500000,23.000000,47.000000,6.260000,0.170000,145.000000
2,2,2009-01-01,2009-01-01,Max.,690.000000,241.000000,7.400000,7.500000,4.800000,33.000000,66.000000,11.200000,0.320000,156.000000
3,3,2009-02-01,2009-02-01,Min.,393.000000,72.000000,6.500000,5.900000,2.800000,20.000000,30.000000,6.820000,0.080000,128.000000
4,4,2009-02-01,2009-02-01,Med.,868.000000,238.000000,7.100000,7.200000,4.100000,21.000000,33.000000,9.550000,0.120000,133.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,571,2024-11-01,2024-11-01,Med.,217.000000,134.000000,7.500000,9.700000,4.200000,39.000000,63.000000,1.570000,0.110000,263.000000
572,572,2024-11-01,2024-11-01,Max.,805.000000,746.000000,7.800000,17.600000,6.000000,45.000000,96.000000,2.800000,0.190000,439.000000
573,573,2024-12-01,2024-12-01,Min.,149.253054,102.219895,7.317452,9.064685,3.245893,45.261261,55.151351,2.388901,0.208919,325.670194
574,574,2024-12-01,2024-12-01,Med.,149.253054,102.219895,7.317452,9.064685,3.245893,45.261261,55.151351,2.388901,0.208919,325.670194


# Mediana

Imputação para colunas com <5% de dados faltantes

In [ ]:
def median_imput(df, metadados, variaveis, max_missing=0.05, 
                 return_reduced=False, save_missing_pct=False, salvar_arquivo=False):
    """
    Faz a imputação dos dados faltantes via mediana.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de interesse.
    max_missing : float, default 0.05
        Determina a fração máxima de dados faltantes tolerados numa coluna.
        Se a fração de dados faltantes for maior que max_missing, o algoritmo de 
        imputação não será aplicado à coluna.
    return_reduced : bool, default False
        Se True, retorna apenas as colunas que receberam imputação.
    save_missing_pct : bool, default False
        Se True, retorna um dicionário com a porcentagem de dados faltantes 
        de cada coluna imputada.
    salvar_arquivo : bool, default False
        Determina se um arquivo Excel com o DataFrame pós imputação deve ser gerado.

    Returns
    -------
    df : pd.DataFrame
        DataFrame pós imputação.
    missing_pct : dict, optional
        Dicionário com as porcentagens de dados faltantes (apenas se save_missing_pct=True).
    """
    df = df.copy()

    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    cols = filtro_colunas(df, min_missing=0, max_missing=max_missing)
    
    # Calcula porcentagem de dados faltantes antes de imputar
    if save_missing_pct:
        missing_pct = df[cols].isna().mean().to_dict()
    
    # Faz a imputação
    df[cols] = df[cols].fillna(df[cols].median())
    
    # Retorna um DataFrame sem as variáveis que não passaram pela imputação
    if return_reduced:
        # Lista os nomes das colunas desejadas no DataFrame final.
        colunas = set(cols + metadados)
        # Busca as colunas presentes na lista anterior.
        colunas = [col for col in df.columns if col in colunas]
        # Reduz o DataFrame às colunas de metadados + variáveis imputadas.
        df = df[colunas]
    
    if salvar_arquivo:
        salvar(df, "imput_median")
    
    if save_missing_pct:
        return df, missing_pct
    
    return df

In [ ]:
# Teste
dados_limpos, metadados, variaveis = dados(r"data\input\dados_limpos.xlsx")

median_imput(dados_limpos, return_reduced=True)

,Index,Data,data_normalizada,Estatistica,COR,TURB.,pH,O.C.,O.D.,Cl,DUR.,Fe,Mn,Cond.
0,0,2009-01-01,2009-01-01,Min.,200.0,64.0,7.0,6.3,2.0,16.0,36.0,3.23,0.09,131.0
1,1,2009-01-01,2009-01-01,Med.,432.0,131.0,7.2,6.8,3.5,23.0,47.0,6.26,0.17,145.0
2,2,2009-01-01,2009-01-01,Max.,690.0,241.0,7.4,7.5,4.8,33.0,66.0,11.20,0.32,156.0
3,3,2009-02-01,2009-02-01,Min.,393.0,72.0,6.5,5.9,2.8,20.0,30.0,6.82,0.08,128.0
4,4,2009-02-01,2009-02-01,Med.,868.0,238.0,7.1,7.2,4.1,21.0,33.0,9.55,0.12,133.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,571,2024-11-01,2024-11-01,Med.,217.0,134.0,7.5,9.7,4.2,39.0,63.0,1.57,0.11,263.0
572,572,2024-11-01,2024-11-01,Max.,805.0,746.0,7.8,17.6,6.0,45.0,96.0,2.80,0.19,439.0
573,573,2024-12-01,2024-12-01,Min.,74.0,21.0,7.3,7.8,3.1,39.0,50.0,1.28,0.15,283.0
574,574,2024-12-01,2024-12-01,Med.,74.0,21.0,7.3,7.8,3.1,39.0,50.0,1.28,0.15,283.0


# KNN

5 - 15% de dados faltantes

In [ ]:
def knn_imput(df, metadados, variaveis, min_missing=0.05, max_missing=0.15, 
              return_reduced=False, save_missing_pct=False, salvar_arquivo=False):
    """
    Faz a imputação dos dados faltantes via KNN (método do sklearn).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame de interesse.
    min_missing : float, default 0.05
        Determina a fração mínima de dados faltantes tolerados numa coluna.
    max_missing : float, default 0.15
        Determina a fração máxima de dados faltantes tolerados numa coluna.
    return_reduced : bool, default False
        Se True, retorna apenas as colunas que receberam imputação.
    save_missing_pct : bool, default False
        Se True, retorna um dicionário com a porcentagem de dados faltantes 
        de cada coluna imputada.
    salvar_arquivo : bool, default False
        Determina se um arquivo Excel com o DataFrame pós imputação deve ser gerado.

    Returns
    -------
    df : pd.DataFrame
        DataFrame pós imputação.
    missing_pct : dict, optional
        Dicionário com as porcentagens de dados faltantes (apenas se save_missing_pct=True).
    """
    df = df.copy()

    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    cols = filtro_colunas(df, min_missing=min_missing, max_missing=max_missing)

    # Calcula porcentagem de dados faltantes antes de imputar
    if save_missing_pct:
        missing_pct = df[cols].isna().mean().to_dict()
    
    imputer = KNNImputer(n_neighbors=2, weights="uniform")

    # Garante valores numéricos
    X = df[cols].astype(float)

    df[cols] = imputer.fit_transform(X)

    # Verifica se ainda há valores faltantes nas colunas imputadas
    print("Valores faltantes após imputação KNN:")
    print(df[cols].isna().sum())

    # Retorna um DataFrame sem as variáveis que não passaram pela imputação
    if return_reduced:
        # Lista os nomes das colunas desejadas no DataFrame final.
        colunas = set(cols + metadados)
        # Busca as colunas presentes na lista anterior.
        colunas = [col for col in df.columns if col in colunas]
        # Reduz o DataFrame às colunas de metadados + variáveis imputadas.
        df = df[colunas]
    
    if salvar_arquivo:
        salvar(df, "imput_knn")

    if save_missing_pct:
        return df, missing_pct

    return df

In [ ]:
# Teste
dados_limpos, metadados, variaveis = dados(r"data\input\dados_limpos.xlsx")

knn_imput(dados_limpos, return_reduced=True)

Valores faltantes após imputação KNN:
ALC.             0
AC.              0
Cianobacteria    0
Clorofila        0
dtype: int64


,Index,Data,data_normalizada,Estatistica,ALC.,AC.,Cianobacteria,Clorofila
0,0,2009-01-01,2009-01-01,Min.,29.000000,3.000000,6600.000000,2.230000
1,1,2009-01-01,2009-01-01,Med.,34.000000,5.000000,6850.000000,11.350000
2,2,2009-01-01,2009-01-01,Max.,38.000000,8.000000,7100.000000,22.320000
3,3,2009-02-01,2009-02-01,Min.,29.000000,6.000000,1100.000000,6.690000
4,4,2009-02-01,2009-02-01,Med.,31.000000,7.000000,2100.000000,7.810000
...,...,...,...,...,...,...,...,...
571,571,2024-11-01,2024-11-01,Med.,48.000000,9.000000,9037.000000,38.680000
572,572,2024-11-01,2024-11-01,Max.,60.000000,18.000000,9037.000000,38.680000
573,573,2024-12-01,2024-12-01,Min.,63.635531,12.540293,10527.398876,36.536836
574,574,2024-12-01,2024-12-01,Med.,63.635531,12.540293,10527.398876,36.536836


# KNN - Escalonamento

15 - 30% de dados faltantes